# RAIQ-200M Tesla T4 Smoke Training

This notebook prepares an evidence-gated RAIQ-200M T4 smoke-v2 experiment on one Google Colab GPU and stores checkpoints on Google Drive. It is an isolated engineering run, not production pretraining.

> Before running: select **Runtime → Change runtime type → T4 GPU**, and upload the two dataset files to `MyDrive/RAIQ/datasets/`.

The current local corpus is an engineering smoke corpus, not a production dataset. A successful run proves pipeline execution only; it does not prove coding/reasoning capability or parity with another model.

In [ ]:
# Cell 1 — Mount Drive and verify the GPU runtime
from google.colab import drive
drive.mount('/content/drive')

import sys, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a Colab GPU runtime first'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('CUDA runtime:', torch.version.cuda)

In [ ]:
# Cell 2 — Clone RAIQ-Core and install without replacing Colab's CUDA PyTorch
%cd /content
!rm -rf RAIQ-Core
!git clone https://github.com/thelegendjamshid456-hash/RAIQ-Core.git
%cd /content/RAIQ-Core
!python -m pip install -e . --no-deps
!python scripts/colab_environment.py

In [ ]:
# Cell 3 — Verify the uploaded dataset and create a Drive manifest
from pathlib import Path
train_path = Path('/content/drive/MyDrive/RAIQ/datasets/technical_toy_train.txt')
validation_path = Path('/content/drive/MyDrive/RAIQ/datasets/technical_toy_validation.txt')
assert train_path.is_file(), f'Missing {train_path}'
assert validation_path.is_file(), f'Missing {validation_path}'
print('Train bytes:', train_path.stat().st_size)
print('Validation bytes:', validation_path.stat().st_size)
assert train_path.stat().st_size == 13860
assert validation_path.stat().st_size == 14547

!mkdir -p /content/drive/MyDrive/RAIQ/manifests /content/drive/MyDrive/RAIQ/tokenizer /content/drive/MyDrive/RAIQ/checkpoints
!python scripts/create_text_manifest.py --train /content/drive/MyDrive/RAIQ/datasets/technical_toy_train.txt --validation /content/drive/MyDrive/RAIQ/datasets/technical_toy_validation.txt --output /content/drive/MyDrive/RAIQ/manifests/technical_smoke_v1.json --corpus-id raiq-technical-smoke-v1 --dataset-version local-inspection-v1 --source-reference gdrive:MyDrive/RAIQ/datasets

In [ ]:
# Cell 4 — Train a BPE tokenizer from training text only
!python scripts/train_bpe_tokenizer.py --input /content/drive/MyDrive/RAIQ/datasets/technical_toy_train.txt --output /content/drive/MyDrive/RAIQ/tokenizer/raiq_code_bpe.json --vocab-size 32768 --name raiq-bpe --version colab-trained-v1 --corpus-id raiq-technical-smoke-v1 --min-pair-frequency 2

from raiq.tokenizer.loader import load_tokenizer
tok = load_tokenizer('/content/drive/MyDrive/RAIQ/tokenizer/raiq_code_bpe.json')
assert tok.vocab_size <= 32768
sample = 'ΔH = m·Cp·ΔT; def solve(x): return x + 1'
assert tok.decode(tok.encode(sample)) == sample
print('Tokenizer vocabulary:', tok.vocab_size)
print('Tokenizer round trip: passed')

In [ ]:
# Cell 5 — Verify the 200M model and one CUDA forward/backward batch
import torch
from raiq.core import RAIQModel, load_config
from raiq.data.manifest import verify_corpus_manifest

cfg = load_config('configs/200m_t4_smoke_v2.yaml')
manifest = verify_corpus_manifest('/content/drive/MyDrive/RAIQ/manifests/technical_smoke_v1.json')
model = RAIQModel(cfg.model).cuda().half()
print('Parameters:', model.parameter_count())
assert model.parameter_count() == 190348032
inputs = torch.randint(0, 384, (1, 2048), device='cuda')
labels = inputs.roll(-1, dims=1)
with torch.autocast('cuda', dtype=torch.float16):
    out = model(inputs, labels=labels)
assert out.loss is not None and torch.isfinite(out.loss)
out.loss.backward()
print('Forward/backward: passed; loss:', float(out.loss))
del model, inputs, labels, out
torch.cuda.empty_cache()

In [ ]:
# Cell 6 — Start a fresh RAIQ-200M T4 smoke-v2 run from step 0
# This command intentionally has no --resume argument and never overwrites the prior run.
# Stop immediately if the model does not fit or any finite check fails.
%cd /content/RAIQ-Core
!python -m raiq.training.train --config configs/200m_t4_smoke_v2.yaml --run-name raiq-200m-t4-smoke-v2 --output-dir /content/drive/MyDrive/RAIQ/checkpoints --max-steps 1000

In [ ]:
# Cell 7 — Inspect fresh smoke-v2 metrics, actual post-clip norms, and GPU memory evidence
import json, torch
metrics_path = '/content/drive/MyDrive/RAIQ/checkpoints/raiq-200m-t4-smoke-v2/metrics.json'
metrics = json.load(open(metrics_path))
validation = [row for row in metrics if 'validation_perplexity' in row]
assert metrics and validation and all('train_loss' in row for row in metrics)
assert max(row['clipped_gradient_norm'] for row in metrics) <= 1.00001
initial_validation = validation[0]
best_validation = min(validation, key=lambda row: row['validation_perplexity'])
final_validation = validation[-1]
assert best_validation['validation_perplexity'] < initial_validation['validation_perplexity']
print('Final record:', metrics[-1])
print('Validation PPL initial/best/final:', initial_validation['validation_perplexity'], best_validation['validation_perplexity'], final_validation['validation_perplexity'])
print('Best validation step:', best_validation['step'])
print('Post-clip norm limit: passed')
print('Peak allocated GiB:', round(torch.cuda.max_memory_allocated() / 2**30, 3))
print('Peak reserved GiB:', round(torch.cuda.max_memory_reserved() / 2**30, 3))

In [ ]:
# Cell 8 — Fresh-run isolation policy
# Do not pass --resume to Cell 6. It starts from step 0 and writes only to
# /content/drive/MyDrive/RAIQ/checkpoints/raiq-200m-t4-smoke-v2/.
# Checkpoint-resume evidence is a separate test after the fresh v2 run completes.

## Completion criteria

Record the actual GPU name, VRAM, peak allocated/reserved memory, tokens per second, raw gradient norm, measured post-clip gradient norm, validation trajectory, checkpoint path, and resume result. Do not claim T4 readiness until these values are observed. This notebook runs only an isolated smoke experiment; it does not start production pretraining.